# Inference 1 video — YOLOv12 + OC-SORT + 2 ROI + 2 garis counting

In [10]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))

PyTorch version: 2.12.0+cu130
CUDA version: 13.0
CUDA available: True
CUDA device name: NVIDIA GeForce RTX 5060 Laptop GPU


In [11]:
# CELL 1 — IMPORT, MODEL, INPUT/OUTPUT
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timedelta
import csv

import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

from boxmot import OcSort as OCSORT

BASE_DIR = Path(r"D:\1-Kuliah\SEKRIPSI\Skripsi_Traffic_Density_Estimation_and_Forecasting")
MODEL_PATH = BASE_DIR/"03_Vehicle_Detection"/"runs"/"train-yolov12s"/"weights"/"best.pt"
VIDEO_PATH = BASE_DIR/"04_Vehicle_Counting"/"video_evaluasi"/"pagi_2menit.mp4"
OUTPUT_DIR = BASE_DIR /"04_Vehicle_Counting"/"evaluasi_traffic_counting" 
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONF = 0.5
IMGSZ = 640
DEVICE = 0 if torch.cuda.is_available() else "cpu"
MAX_FRAMES = None  # Isi 300 untuk tes cepat; None = seluruh video

# WAJIB sesuaikan dengan tanggal/jam frame PERTAMA rekaman CCTV.
VIDEO_START = "2026-04-10 06:00:00"  # CONTOH, BUKAN waktu otomatis dari file video
COUNT_INTERVAL_SEC = 10  # Ubah menjadi 15 * 60 untuk interval 15 menit
assert COUNT_INTERVAL_SEC > 0, "COUNT_INTERVAL_SEC harus lebih besar dari nol"
SHOW_POLYGON = True
SHOW_LINE = True
SHOW_TABLE = True
SAVE_VIDEO = True  # False = hanya CSV, True = video + CSV

assert MODEL_PATH.is_file(), f"Model tidak ditemukan: {MODEL_PATH}"
assert VIDEO_PATH.is_file(), f"Video tidak ditemukan: {VIDEO_PATH}"
model = YOLO(str(MODEL_PATH))
class_names = model.names  # ID kelas selalu dibaca dari checkpoint, bukan diasumsikan dari YAML.
valid_classes = {"motor", "mobil", "bus", "truk"}
print("Nama kelas dari best.pt:", class_names)
print("Input:", VIDEO_PATH)
print("Output:", OUTPUT_DIR)

Nama kelas dari best.pt: {0: 'bus', 1: 'mobil', 2: 'motor', 3: 'truk'}
Input: D:\1-Kuliah\SEKRIPSI\Skripsi_Traffic_Density_Estimation_and_Forecasting\04_Vehicle_Counting\video_evaluasi\pagi_2menit.mp4
Output: D:\1-Kuliah\SEKRIPSI\Skripsi_Traffic_Density_Estimation_and_Forecasting\04_Vehicle_Counting\evaluasi_traffic_counting


In [12]:
# CELL 2 — POLIGON + GARIS TERPISAH (KOORDINAT ACUAN 1280×720)
# Sesuaikan POLYGON dan LINE dari frame video yang benar.
REF_WIDTH, REF_HEIGHT = 1280, 720

ROI_CONFIG = {
    "IN": {
        "polygon": [(497, 234),(40, 280),(124, 718), (1181, 535)],
        "line": ((106, 619), (901, 419)),
        "direction": "up",  # notebook referensi: IN y menurun; ubah "down" bila sebaliknya
        "color": (0, 255, 255),  # kuning (BGR)
    },
    "OUT": {
        "polygon": [(642, 263), (1102, 144), (1265, 172), (1261, 511)],
        "line": ((1024, 414), (1215, 167)),
        "direction": "down",  # notebook referensi: OUT y naik; ubah "up" bila sebaliknya
        "color": (0, 0, 255),  # merah (BGR)
    },
}

# ROI dievaluasi dengan TITIK BAWAH TENGAH bbox, bukan seluruh luas bbox.
# Jika koordinat video bukan 1280x720, titik otomatis diskalakan sesuai resolusinya.
def scale_point(point, sx, sy):
    return (int(round(point[0] * sx)), int(round(point[1] * sy)))


def prepare_rois(width, height):
    sx, sy = width / REF_WIDTH, height / REF_HEIGHT
    rois = {}
    for name, cfg in ROI_CONFIG.items():
        polygon = np.array([scale_point(p, sx, sy) for p in cfg["polygon"]], dtype=np.int32)
        line = tuple(scale_point(p, sx, sy) for p in cfg["line"])
        assert cfg["direction"] in {"up", "down"}, f"Arah {name} harus up/down"
        assert len(polygon) >= 3, f"ROI {name} harus memiliki >=3 titik"
        for endpoint in line:
            assert cv2.pointPolygonTest(polygon, endpoint, False) >= 0, (
                f"Ujung garis {name} {endpoint} berada di luar polygon; atur ulang LINE."
            )
        rois[name] = {"polygon": polygon, "line": line,
                      "direction": cfg["direction"], "color": cfg["color"]}
    return rois

In [13]:
# CELL 3 — FUNGSI ROI DAN SEGMENT-CROSSING
def bottom_center(box):
    x1, y1, x2, y2 = box
    return (int(round((x1 + x2) / 2)), int(round(y2)))


def get_zone(point, rois):
    # Prioritas IN lalu OUT apabila poligon sengaja dibuat saling bertumpuk.
    for zone in ("IN", "OUT"):
        if cv2.pointPolygonTest(rois[zone]["polygon"], point, False) >= 0:
            return zone
    return None


def cross(a, b, p):
    return (b[0] - a[0]) * (p[1] - a[1]) - (b[1] - a[1]) * (p[0] - a[0])


def crossed_finite_line(previous, current, line):
    """Pergerakan menyeberangi segmen counting (kedua ujung garis terbatas)."""
    a, b = line
    side_prev = cross(a, b, previous)
    side_curr = cross(a, b, current)
    if not (side_prev < 0 < side_curr or side_curr < 0 < side_prev):
        return False
    # Garis boleh benar-benar berpotongan hanya jika titik ujung counting
    # juga berada di dua sisi yang berbeda terhadap lintasan kendaraan.
    end_a = cross(previous, current, a)
    end_b = cross(previous, current, b)
    return end_a * end_b <= 0


def correct_direction(previous, current, direction):
    dy = current[1] - previous[1]  # OpenCV: y semakin besar berarti ke bawah.
    return dy < 0 if direction == "up" else dy > 0


def yolo_detections_in_roi(result, rois):
    """Hasil YOLO Nx6 hanya kendaraan yang bottom-center masuk IN/OUT."""
    detections = []
    boxes = result.boxes
    if boxes is None:
        return np.empty((0, 6), dtype=np.float32)
    for box, confidence, cls in zip(
        boxes.xyxy.cpu().numpy(), boxes.conf.cpu().numpy(), boxes.cls.cpu().numpy()
    ):
        cls_id = int(cls)
        name = str(class_names.get(cls_id, "unknown")).strip().lower()
        if confidence < CONF or name not in valid_classes:
            continue
        if get_zone(bottom_center(box), rois) is None:
            continue
        detections.append([*box.tolist(), float(confidence), cls_id])
    return np.asarray(detections, dtype=np.float32).reshape(-1, 6)


def draw_static_roi(frame, rois):
    for zone, cfg in rois.items():
        color = cfg["color"]
        if SHOW_POLYGON:
            cv2.polylines(frame, [cfg["polygon"]], True, color, 2)
        if SHOW_LINE:
            cv2.line(frame, *cfg["line"], color, 3)
            cv2.putText(frame, f"LINE {zone}", cfg["line"][0],
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)


def draw_counts(frame, counts):
    """Tabel counting 3 baris x 5 kolom: label IN kuning dan OUT merah."""
    if not SHOW_TABLE:
        return

    height, width = frame.shape[:2]
    margin = 12
    table_width = min(440, width - 2 * margin)
    table_height = min(105, height - 2 * margin)
    if table_width < 150 or table_height < 65:
        return  # Frame terlalu kecil untuk tabel.

    x0, y0 = (width - table_width) // 2, 16
    x_edges = [x0 + round(i * table_width / 5) for i in range(6)]
    y_edges = [y0 + round(i * table_height / 3) for i in range(4)]
    cyan = (255, 230, 0)       # BGR: garis biru cyan
    white = (255, 255, 255)
    label_colors = {
        "IN": (0, 255, 255),  # BGR: kuning
        "OUT": (0, 0, 255),   # BGR: merah
    }

    # Latar abu-abu gelap transparan: video masih tampak samar di balik tabel.
    roi = frame[y0:y_edges[-1], x0:x_edges[-1]]
    background = np.full_like(roi, (45, 45, 45))
    cv2.addWeighted(background, 0.82, roi, 0.18, 0, dst=roi)

    # Bingkai + pemisah sel seperti contoh (5 kolom x 3 baris).
    thickness = max(2, round(table_width / 220))
    cv2.rectangle(frame, (x_edges[0], y_edges[0]),
                  (x_edges[-1], y_edges[-1]), cyan, thickness)
    for x in x_edges[1:-1]:
        cv2.line(frame, (x, y_edges[0]), (x, y_edges[-1]), cyan, thickness)
    for y in y_edges[1:-1]:
        cv2.line(frame, (x_edges[0], y), (x_edges[-1], y), cyan, thickness)

    columns = ("", "mobil", "motor", "bus", "truk")
    rows = [columns]
    for zone in ("IN", "OUT"):
        rows.append((zone, *(str(counts[zone][name]) for name in columns[1:])))

    font = cv2.FONT_HERSHEY_SIMPLEX
    for row_idx, cells in enumerate(rows):
        for col_idx, value in enumerate(cells):
            if not value:
                continue
            left, right = x_edges[col_idx], x_edges[col_idx + 1]
            top, bottom = y_edges[row_idx], y_edges[row_idx + 1]
            cell_w, cell_h = right - left, bottom - top

            # Pastikan teks tetap di tengah sel dan muat saat resolusi mengecil.
            font_scale = min(0.92, cell_h / 61)
            (text_w, text_h), baseline = cv2.getTextSize(value, font, font_scale, 2)
            if text_w > cell_w * 0.82 or text_h > cell_h * 0.67:
                font_scale *= min(cell_w * 0.82 / max(text_w, 1),
                                  cell_h * 0.67 / max(text_h, 1))
            text_thickness = 2 if font_scale >= 0.58 else 1
            (text_w, text_h), baseline = cv2.getTextSize(
                value, font, font_scale, text_thickness
            )
            text_x = left + (cell_w - text_w) // 2
            text_y = top + (cell_h + text_h) // 2
            color = label_colors[value] if col_idx == 0 else white
            cv2.putText(frame, value, (text_x, text_y), font, font_scale,
                        color, text_thickness, cv2.LINE_AA)


In [14]:

# CELL 4 — INFERENCE SATU VIDEO + CSV INTERVAL


# PENGATURAN OUTPUT
PROGRESS_EVERY = 100

CSV_CLASSES = ("motor", "mobil", "bus", "truk")

CSV_COLUMNS = [
    "Date", "Time", "session", "day_type",
    "motor_IN", "mobil_IN", "bus_IN", "truk_IN",
    "motor_OUT", "mobil_OUT", "bus_OUT", "truk_OUT",
]



# 1. INISIALISASI COUNTING
def fresh_counts():
    return {
        zone: {cls: 0 for cls in CSV_CLASSES}
        for zone in ("IN", "OUT")
    }



# 2. MEMBUAT BARIS CSV PER INTERVAL
def interval_row(start_datetime, bucket_number, bucket_counts):

    timestamp = start_datetime + timedelta(
        seconds=bucket_number * COUNT_INTERVAL_SEC
    )

    hour = timestamp.hour

    session = (
        "pagi" if 6 <= hour < 10
        else "sore" if 15 <= hour < 19
        else "di_luar_sesi"
    )

    day_type = 1 if timestamp.weekday() >= 5 else 0

    return [
        timestamp.strftime("%Y-%m-%d"),
        f"{timestamp.hour}:{timestamp:%M:%S}",
        session,
        day_type,
        *(bucket_counts["IN"][cls] for cls in CSV_CLASSES),
        *(bucket_counts["OUT"][cls] for cls in CSV_CLASSES),
    ]



# 3. INFERENCE SATU VIDEO
def run_single_video():

    # -------------------------------------------------
    # A. INFORMASI VIDEO
    # -------------------------------------------------

    start_datetime = datetime.strptime(
        VIDEO_START, "%Y-%m-%d %H:%M:%S"
    )

    cap = cv2.VideoCapture(str(VIDEO_PATH))

    assert cap.isOpened(), (
        f"Gagal membuka video: {VIDEO_PATH}"
    )

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if not fps or not np.isfinite(fps) or fps <= 0:
        fps = 25.0
        print("PERINGATAN: FPS tidak tersedia. Menggunakan 25 FPS.")

    rois = prepare_rois(width, height)

    hour = start_datetime.hour

    session = (
        "pagi" if 6 <= hour < 10
        else "sore" if 15 <= hour < 19
        else "di_luar_sesi"
    )

    # -------------------------------------------------
    # B. INISIALISASI TRACKER
    # -------------------------------------------------

    tracker = OCSORT(
        det_thresh=CONF,
        max_age=30,
        min_hits=2,
        iou_threshold=0.3,
        per_class=True
    )

    # -------------------------------------------------
    # C. OUTPUT
    # -------------------------------------------------

    output_video = (
        OUTPUT_DIR / f"{VIDEO_PATH.stem}_roi_counting.mp4"
    )

    output_csv = (
        OUTPUT_DIR / f"{VIDEO_PATH.stem}_interval_counting.csv"
    )

    writer = None

    if SAVE_VIDEO:

        writer = cv2.VideoWriter(
            str(output_video),
            cv2.VideoWriter_fourcc(*"mp4v"),
            fps,
            (width, height)
        )

        if not writer.isOpened():
            cap.release()
            raise RuntimeError(
                f"Gagal membuat video output: {output_video}"
            )

    # -------------------------------------------------
    # D. TAMPILKAN INFORMASI VIDEO
    # -------------------------------------------------

    print(f"\n📹 Video      : {VIDEO_PATH}")
    print(f"🕒 Start Time : {start_datetime}")
    print(f"🧭 Session    : {session}")
    print(f"📐 Resolusi   : {width}x{height}")
    print(f"🎞 FPS        : {fps}")
    print(f"⏱ Interval   : {COUNT_INTERVAL_SEC} detik")

    print("\n" + "=" * 60)
    print("MEMULAI PROSES INFERENCE")
    print("=" * 60 + "\n")

    # -------------------------------------------------
    # E. INISIALISASI VARIABEL COUNTING
    # -------------------------------------------------

    total_counts = fresh_counts()
    interval_counts = fresh_counts()

    counted_ids = {
        "IN": set(),
        "OUT": set()
    }

    last_seen = {}

    frame_index = 0
    interval_index = 0
    rows_written = 0

    # -------------------------------------------------
    # F. PROSES VIDEO DAN CSV
    # -------------------------------------------------

    try:

        with output_csv.open(
            "w",
            newline="",
            encoding="utf-8"
        ) as csv_file:

            csv_writer = csv.writer(csv_file)

            csv_writer.writerow(CSV_COLUMNS)
            csv_file.flush()

            try:

                while True:

                    # ---------------------------------
                    # BATAS FRAME
                    # ---------------------------------

                    if (
                        MAX_FRAMES is not None
                        and frame_index >= MAX_FRAMES
                    ):
                        break

                    ok, frame = cap.read()

                    if not ok:
                        break

                    # ---------------------------------
                    # WAKTU VIDEO DAN INTERVAL
                    # ---------------------------------

                    video_seconds = frame_index / fps

                    target_interval = int(
                        video_seconds // COUNT_INTERVAL_SEC
                    )

                    while target_interval > interval_index:

                        csv_writer.writerow(
                            interval_row(
                                start_datetime,
                                interval_index,
                                interval_counts
                            )
                        )

                        csv_file.flush()

                        rows_written += 1
                        interval_index += 1

                        # Reset jumlah interval saja.
                        interval_counts = fresh_counts()

                    frame_index += 1

                    # =================================
                    # TAHAP 1 — YOLO INFERENCE
                    # =================================

                    result = model.predict(
                        frame,
                        conf=CONF,
                        imgsz=IMGSZ,
                        device=DEVICE,
                        half=(DEVICE != "cpu"),
                        verbose=False
                    )[0]

                    # Filter kendaraan dalam ROI.
                    detections = yolo_detections_in_roi(
                        result,
                        rois
                    )

                    # =================================
                    # TAHAP 2 — OC-SORT TRACKING
                    # =================================

                    tracks = tracker.update(
                        detections,
                        frame
                    )

                    # Gambar polygon hanya jika video disimpan.
                    if SAVE_VIDEO:
                        draw_static_roi(frame, rois)

                    # =================================
                    # TAHAP 3 — VEHICLE COUNTING
                    # =================================

                    if tracks is not None:

                        for trk in tracks:

                            if len(trk) < 7:
                                continue

                            if len(trk) >= 8 and int(trk[7]) == -1:
                                continue

                            x1, y1, x2, y2 = [
                                int(round(v))
                                for v in trk[:4]
                            ]

                            track_id = int(trk[4])
                            cls_id = int(trk[6])

                            class_name = str(
                                class_names.get(
                                    cls_id,
                                    "unknown"
                                )
                            ).strip().lower()

                            if class_name not in valid_classes:
                                continue

                            # Titik bawah tengah kendaraan.
                            point = bottom_center(
                                (x1, y1, x2, y2)
                            )

                            # Tentukan jalur IN atau OUT.
                            zone = get_zone(point, rois)

                            if zone is None:
                                continue

                            cfg = rois[zone]

                            key = (cls_id, track_id)

                            previous = last_seen.get(key)

                            # -------------------------
                            # DETEKSI CROSSING
                            # -------------------------

                            if previous is not None:

                                prev_point, prev_frame, prev_zone = previous

                                if (
                                    zone == prev_zone

                                    and frame_index - prev_frame <= 5

                                    and cv2.pointPolygonTest(
                                        cfg["polygon"],
                                        prev_point,
                                        False
                                    ) >= 0

                                    and key not in counted_ids[zone]

                                    and crossed_finite_line(
                                        prev_point,
                                        point,
                                        cfg["line"]
                                    )

                                    and correct_direction(
                                        prev_point,
                                        point,
                                        cfg["direction"]
                                    )
                                ):

                                    # Total kumulatif.
                                    total_counts[zone][class_name] += 1

                                    # Jumlah interval saat ini.
                                    interval_counts[zone][class_name] += 1

                                    # Hindari double counting.
                                    counted_ids[zone].add(key)

                            # -------------------------
                            # UPDATE POSISI TRACKER
                            # -------------------------

                            if (
                                cross(
                                    *cfg["line"],
                                    point
                                ) != 0

                                or previous is None

                                or zone != previous[2]
                            ):

                                last_seen[key] = (
                                    point,
                                    frame_index,
                                    zone
                                )

                            # -------------------------
                            # VISUALISASI (OPSIONAL)
                            # -------------------------

                            if SAVE_VIDEO:

                                color = cfg["color"]

                                cv2.rectangle(
                                    frame,
                                    (x1, y1),
                                    (x2, y2),
                                    color,
                                    2
                                )

                                cv2.circle(
                                    frame,
                                    point,
                                    4,
                                    color,
                                    -1
                                )

                                label = (
                                    f"{zone} {class_name} "
                                    f"ID:{track_id}"
                                )

                                cv2.putText(
                                    frame,
                                    label,
                                    (x1, max(22, y1 - 8)),
                                    cv2.FONT_HERSHEY_SIMPLEX,
                                    0.55,
                                    color,
                                    2
                                )

                    # =================================
                    # TAHAP 4 — SIMPAN FRAME (OPSIONAL)
                    # =================================

                    if SAVE_VIDEO:

                        draw_counts(frame, total_counts)

                        writer.write(frame)

                    # =================================
                    # TAHAP 5 — PROGRESS SETIAP 100 FRAME
                    # =================================

                    if frame_index % PROGRESS_EVERY == 0:

                        elapsed_seconds = frame_index / fps

                        elapsed_time = str(
                            timedelta(
                                seconds=int(elapsed_seconds)
                            )
                        )

                        current_time = (
                            start_datetime
                            + timedelta(seconds=elapsed_seconds)
                        )

                        print(
                            f"Memproses Frame: {frame_index:,} | "
                            f"Durasi: {elapsed_time} | "
                            f"Waktu CCTV: {current_time:%H:%M:%S}",
                            flush=True
                        )

            finally:

                # -------------------------------------
                # SIMPAN INTERVAL TERAKHIR
                # -------------------------------------

                if frame_index > 0:

                    csv_writer.writerow(
                        interval_row(
                            start_datetime,
                            interval_index,
                            interval_counts
                        )
                    )

                    csv_file.flush()

                    rows_written += 1

    finally:

        cap.release()

        if writer is not None:
            writer.release()

    # =================================================
    # G. REKAP HASIL COUNTING
    # =================================================

    summary = pd.DataFrame([
        {
            "Arah": zone,
            **total_counts[zone],
            "Total": sum(total_counts[zone].values())
        }
        for zone in ("IN", "OUT")
    ])

    # =================================================
    # H. INFORMASI HASIL AKHIR
    # =================================================

    duration = timedelta(
        seconds=int(frame_index / fps)
    )

    print("\n" + "=" * 60)
    print("=== SELESAI ===")
    print("=" * 60)

    if SAVE_VIDEO:
        print(f"Video        : {output_video}")

    print(f"CSV          : {output_csv}")
    print(f"Interval     : {COUNT_INTERVAL_SEC} detik")
    print(f"Baris CSV    : {rows_written}")
    print(f"Durasi Video : {duration}")

    print("\nHASIL VEHICLE COUNTING")
    print(summary.to_string(index=False))

    print("=" * 60)

    return summary

In [15]:
# CELL 5 — JALANKAN SEKALI (HANYA MENYIMPAN FILE)
summary = run_single_video()

INFO     OcSort: det_thresh=0.5, max_age=30, max_obs=50, min_hits=2, iou_threshold=0.3, per_class=True,            
         class_ids=None, asso_func=iou, variable_dt=False


📹 Video      : D:\1-Kuliah\SEKRIPSI\Skripsi_Traffic_Density_Estimation_and_Forecasting\04_Vehicle_Counting\video_evaluasi\pagi_2menit.mp4
🕒 Start Time : 2026-04-10 06:00:00
🧭 Session    : pagi
📐 Resolusi   : 1280x720
🎞 FPS        : 25.01036226477659
⏱ Interval   : 10 detik

MEMULAI PROSES INFERENCE

Memproses Frame: 100 | Durasi: 0:00:03 | Waktu CCTV: 06:00:03
Memproses Frame: 200 | Durasi: 0:00:07 | Waktu CCTV: 06:00:07
Memproses Frame: 300 | Durasi: 0:00:11 | Waktu CCTV: 06:00:11
Memproses Frame: 400 | Durasi: 0:00:15 | Waktu CCTV: 06:00:15
Memproses Frame: 500 | Durasi: 0:00:19 | Waktu CCTV: 06:00:19
Memproses Frame: 600 | Durasi: 0:00:23 | Waktu CCTV: 06:00:23
Memproses Frame: 700 | Durasi: 0:00:27 | Waktu CCTV: 06:00:27
Memproses Frame: 800 | Durasi: 0:00:31 | Waktu CCTV: 06:00:31
Memproses Frame: 900 | Durasi: 0:00:35 | Waktu CCTV: 06:00:35
Memproses Frame: 1,000 | Durasi: 0:00:39 | Waktu CCTV: 06:00:39
Memproses Frame: 1,100 | Durasi: 0:00:43 | Waktu CCTV: 06:00:43
Memproses Fr

In [16]:
# EVALUASI VEHICLE COUNTING - Membandingkan hasil counting kendaraan manual dengan counting otomatis (sistem)


import pandas as pd
import numpy as np
from pathlib import Path

# 1. Lokasi file CSV
BASE_DIR = Path(r"D:\1-Kuliah\SEKRIPSI\Skripsi_Traffic_Density_Estimation_and_Forecasting")
VIDEO_PATH = BASE_DIR/"04_Vehicle_Counting"/"video_evaluasi"/"pagi_2menit.mp4"
INPUT_DIR = VIDEO_PATH.parent / "output_roi_counting" / "untuk_evaluasi"

FILE_MANUAL = INPUT_DIR / "manual_counting_pagi_2menit_interval_counting.csv"
FILE_OTOMATIS = INPUT_DIR / "pagi_2menit_interval_counting.csv"

# 2. Membaca CSV
manual = pd.read_csv(FILE_MANUAL)
otomatis = pd.read_csv(FILE_OTOMATIS)

# 3. Kolom yang akan dievaluasi
kolom_kendaraan = [
    "motor_IN", "mobil_IN", "bus_IN", "truk_IN",
    "motor_OUT", "mobil_OUT", "bus_OUT", "truk_OUT"
]

# 4. Mengambil total kendaraan
# Manual: hanya 1 baris total
# Otomatis: jumlah seluruh interval 10 detik

total_manual = manual[kolom_kendaraan].sum()
total_otomatis = otomatis[kolom_kendaraan].sum()

# 5. Menghitung MAE, MAPE, dan Counting Accuracy
hasil = []

for kolom in kolom_kendaraan:

    aktual = int(total_manual[kolom])
    prediksi = int(total_otomatis[kolom])

    # MAE (1 pengamatan total per kelas-arah)
    mae = abs(aktual - prediksi)

    if aktual > 0:
        mape = (mae / aktual) * 100
        accuracy = (1 - mae / aktual) * 100
    else:
        mape = np.nan
        accuracy = np.nan

    hasil.append({
        "Kelas": kolom,
        "Manual": aktual,
        "Otomatis": prediksi,
        "MAE": mae,
        "MAPE (%)": round(mape, 2),
        "Counting Accuracy (%)": round(accuracy, 2)
    })

# 6. Menampilkan hasil per kelas dan arah
df_hasil = pd.DataFrame(hasil)

print("HASIL EVALUASI PER KELAS DAN ARAH")
display(df_hasil)

# 7. Ringkasan keseluruhan
total_aktual = df_hasil["Manual"].sum()
total_prediksi = df_hasil["Otomatis"].sum()
total_error = df_hasil["MAE"].sum()

# Rata-rata MAE dari 8 kelompok kelas-arah
mae_keseluruhan = df_hasil["MAE"].mean()

# MAPE rata-rata pada kelompok dengan aktual > 0
mape_keseluruhan = df_hasil["MAPE (%)"].mean()

# Accuracy berbasis total kesalahan absolut
accuracy_keseluruhan = (
    (1 - total_error / total_aktual) * 100
    if total_aktual > 0 else np.nan
)

print("\nRINGKASAN KESELURUHAN")
print("Total Manual      :", total_aktual)
print("Total Otomatis    :", total_prediksi)
print("Total Error       :", total_error)
print("MAE               :", round(mae_keseluruhan, 4))
print("MAPE (%)          :", round(mape_keseluruhan, 2))
print("Counting Accuracy :", round(accuracy_keseluruhan, 2), "%")

# 8. Menyimpan hasil evaluasi
OUTPUT_FILE = INPUT_DIR / "hasil_evaluasi.csv"

df_hasil.to_csv(OUTPUT_FILE, index=False)

print("\nHasil disimpan di:", OUTPUT_FILE)


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\1-Kuliah\\SEKRIPSI\\Skripsi_Traffic_Density_Estimation_and_Forecasting\\04_Vehicle_Counting\\video_evaluasi\\output_roi_counting\\untuk_evaluasi\\manual_counting_pagi_2menit_interval_counting.csv'